# <center>Graduation Project: SQL</center>

**Task description and research goal:**
After the company acquired a subscription-based book reading service, we need to analyze its database. It contains information about books, publishers and authors, as well as user reviews of books. This data will help us shape a value proposition for the new product.

**Data description**

Table **books**
Contains data about books:
 - book_id - book identifier;
 - author_id - author identifier;
 - title - book title;
 - num_pages - number of pages;
 - publication_date - book publication date;
 - publisher_id - publisher identifier.

Table **authors**
Contains data about authors:
 - author_id - author identifier;
 - author - author name.

Table **publishers**
Contains data about publishers:
 - publisher_id - publisher identifier;
 - publisher - publisher name.

Table **ratings**
Contains user ratings of books:
 - rating_id - rating identifier;
 - book_id - book identifier;
 - username - name of the user who left the rating;
 - rating - book rating.

Table **reviews**
Contains user reviews of books:
 - review_id - review identifier;
 - book_id - book identifier;
 - username - name of the user who wrote the review;
 - text - review text.

**Project plan**
1. Load libraries, configure parameters and connect to the database
2. Count how many books were released after January 1, 2000
3. For each book, count the number of reviews and the average rating
4. Find the publisher that released the largest number of books with more than 50 pages — this filters out brochures
5. Find the author with the highest average book rating — only consider books with 50 or more ratings
6. Compute the average number of reviews from users who left more than 48 ratings


In [ ]:
# import libraries
import pandas as pd
import sqlalchemy as sa


In [ ]:
# connection parameters (training credentials provided by the course;
# in a real project these would come from environment variables or a secrets manager)
db_config = {
    'user': 'praktikum_student',                              # username
    'pwd': 'Sdf4$2;d-d30pp',                                  # password
    'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',      # host
    'port': 6432,                                             # port
    'db': 'data-analyst-final-project-db',                    # database name
}
connection_string = (
    'postgresql://{user}:{pwd}@{host}:{port}/{db}'.format(**db_config)
)

# create the connector
engine = sa.create_engine(
    connection_string, connect_args={'sslmode': 'require'}
)


In [ ]:
# helper function for running SQL queries via pandas
def get_sql_data(
    query: str, engine: sa.engine.base.Engine = engine
) -> pd.DataFrame:
    """Open a connection, fetch data from SQL, close the connection."""
    with engine.connect() as con:
        return pd.read_sql(sql=sa.text(query), con=con)


### Exploring the Data

**books**

In [ ]:
query = """
SELECT *
FROM books
LIMIT 5
"""
get_sql_data(query)


In [ ]:
query = """
SELECT COUNT(*)
FROM books
"""
get_sql_data(query)


In [ ]:
query = """
SELECT MIN(publication_date),
       MAX(publication_date)
FROM books
"""
get_sql_data(query)


We can see that the database contains information about 1,000 books published between 1952-12-01 and 2020-03-31.

**authors**

In [ ]:
query = """
SELECT *
FROM authors
LIMIT 5
"""
get_sql_data(query)


In [ ]:
query = """
SELECT COUNT(*)
FROM authors
"""
get_sql_data(query)


There are 636 authors in the database.

**ratings**

In [ ]:
query = """
SELECT *
FROM ratings
LIMIT 5
"""
get_sql_data(query)


In [ ]:
query = """
SELECT COUNT(*)
FROM ratings
"""
get_sql_data(query)


There are 6,456 ratings in the database.

**reviews**

In [ ]:
query = """
SELECT *
FROM reviews
LIMIT 5
"""
get_sql_data(query)


In [ ]:
query = """
SELECT COUNT(*)
FROM reviews
"""
get_sql_data(query)


There are 2,793 book reviews.

**publishers**

In [ ]:
query = """
SELECT *
FROM publishers
LIMIT 5
"""
get_sql_data(query)


In [ ]:
query = """
SELECT COUNT(*)
FROM publishers
"""
get_sql_data(query)


There are 340 publishers.

Based on the data above, we can conclude:
- The `books` table contains 1,000 books published between 1952-12-01 and 2020-03-31.
- The `authors` table contains 636 authors.
- The `publishers` table contains 340 publishers.
- The `ratings` table contains 6,456 book ratings.
- The `reviews` table contains 2,793 book reviews.

Users actively rate and comment on books. 6,456 ratings and 2,793 reviews is enough data to be useful for building user recommendations.

The dataset contains a fairly large number of books, authors and publishers, which means it is representative enough to be used to analyze a wide range of book-related questions.

### How many books were released after January 1, 2000

In [ ]:
query = """
SELECT COUNT(*) AS books_after_2000
FROM books
WHERE publication_date > CAST('2000-01-01' AS DATE)
"""
get_sql_data(query)


819 books were released after January 1, 2000.

### Number of reviews and average rating for each book

In [ ]:
query = """
SELECT b.title                              AS book_title,
       COUNT(DISTINCT r.review_id)          AS review_count,
       ROUND(AVG(rat.rating)::NUMERIC, 2)   AS average_rating
FROM books AS b
LEFT JOIN reviews AS r
       ON b.book_id = r.book_id
LEFT JOIN ratings AS rat
       ON b.book_id = rat.book_id
GROUP BY b.book_id, b.title
ORDER BY average_rating DESC NULLS LAST
"""
get_sql_data(query)


The query result is a table with the book title, the number of reviews and the average rating for each book. `COUNT(DISTINCT review_id)` is used because the join with `ratings` would otherwise inflate the review count.

### Publisher that released the largest number of books with more than 50 pages

In [ ]:
query = """
WITH top_publisher AS (
    SELECT p.publisher        AS publisher_name,
           COUNT(b.book_id)   AS books_count
    FROM publishers AS p
    JOIN books AS b
      ON p.publisher_id = b.publisher_id
    WHERE b.num_pages > 50
    GROUP BY p.publisher
    ORDER BY books_count DESC
    LIMIT 1
)
SELECT publisher_name,
       books_count
FROM top_publisher
"""
get_sql_data(query)


Penguin Books is the publisher that released the largest number of books with more than 50 pages — 42 books.

### Author with the highest average book rating (only counting books with 50 or more ratings)

In [ ]:
query = """
WITH top_count_rat AS (
    SELECT book_id,
           COUNT(rating_id)   AS ratings_count,
           AVG(rating)        AS avg_rat
    FROM ratings
    GROUP BY book_id
    HAVING COUNT(rating_id) >= 50
),
author_book AS (
    SELECT b.author_id,
           a.author,
           AVG(t.avg_rat)     AS avg_author_rating
    FROM books AS b
    JOIN authors AS a
      ON b.author_id = a.author_id
    JOIN top_count_rat AS t
      ON b.book_id = t.book_id
    GROUP BY b.author_id, a.author
    ORDER BY avg_author_rating DESC
    LIMIT 1
)
SELECT *
FROM author_book
"""
get_sql_data(query)


According to the data, books by J.K. Rowling / Mary GrandPré have the highest average rating — 4.28. Only books that received at least 50 ratings were considered in the analysis.

### Average number of reviews from users who left more than 48 ratings

In [ ]:
query = """
WITH active_raters AS (
    SELECT username,
           COUNT(rating_id) AS ratings_count
    FROM ratings
    GROUP BY username
    HAVING COUNT(rating_id) > 48
),
user_review_counts AS (
    SELECT ar.username,
           COUNT(r.review_id) AS reviews_count
    FROM active_raters AS ar
    LEFT JOIN reviews AS r
           ON ar.username = r.username
    GROUP BY ar.username
)
SELECT ROUND(AVG(reviews_count), 2) AS avg_reviews_per_active_user
FROM user_review_counts
"""
get_sql_data(query)


The average number of reviews from users who left more than 48 ratings is approximately 24.

### Final Conclusion

The analysis used 5 tables and ran 5 SQL queries to answer the research questions.

**Key findings**

- 819 books were released after January 1, 2000 — i.e. roughly 82% of the catalog is post-2000, which means the catalog is dominated by relatively modern titles.
- For every book, we computed the number of reviews and the average rating. The result is a table with the book title, review count and average rating per book.
- Penguin Books is the publisher with the largest number of books with more than 50 pages — 42 books.
- The author with the highest average rating (across books with 50+ ratings) is J.K. Rowling / Mary GrandPré, with an average rating of 4.28.
- Among users who left more than 48 ratings, the average number of reviews per user is about 24.

**Business recommendations**

- *Catalog focus.* Since most of the catalog (≈82%) consists of books published after 2000, marketing communications and editorial recommendations should be tuned to a contemporary audience. Books published before 2000 can be presented as a separate "classics" sub-collection rather than mixed in with new releases.
- *Highlight strong publishers.* Penguin Books contributes the largest share of long-form titles (>50 pages) in the catalog. It is worth featuring publisher-led selections ("Best of Penguin", "Editor's Choice") to leverage existing brand recognition and improve discovery of catalog depth.
- *Promote top-rated authors.* Authors such as J.K. Rowling / Mary GrandPré, with consistently high ratings on books with a meaningful number of votes, should be used as an anchor for personalized recommendations and onboarding lists for new subscribers.
- *Engage power users.* Users who left more than 48 ratings produce on average ~24 reviews each — they are clearly an engaged minority. They should be a priority audience for a beta program, an early-access feature, or a referral incentive: their reviews drive content discovery for the rest of the user base.
- *Encourage reviews from average users.* Most users rate but do not write a full review. Lightweight in-app prompts (e.g. "What did you like about this book?" right after rating) could meaningfully increase the volume of qualitative feedback the recommendation engine can rely on.